# Real-Time Inference Engine Validation

Notebook ini memvalidasi alur real-time inference tanpa menambahkan file deep learning `.py`: live buffer sufficiency, online feature sequence, LSTM checkpoint inference, inverse transform ke km/h, data-quality metadata, confidence adjustment, dan response-time target.

In [1]:
from __future__ import annotations

import json
import sys
import time
from dataclasses import asdict, dataclass
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "traffic_prediction").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if not SRC.exists():
    raise RuntimeError(f"Could not locate project src directory from {Path.cwd()}")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from traffic_prediction.config.settings import load_config
from traffic_prediction.data.scalers import ScalerStore
from traffic_prediction.data.schemas import FeatureManifest, LiveTrafficRecord
from traffic_prediction.features.online import OnlineFeatureEngineer
from traffic_prediction.inference.confidence import ConfidenceAdjuster
from traffic_prediction.inference.congestion import classify_congestion
from traffic_prediction.ingestion.buffer import LiveBufferManager
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM
from traffic_prediction.monitoring.data_quality import DataQualityMonitor

config = load_config(project_root=PROJECT_ROOT)
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "models"
REPORT_ROOT = PROJECT_ROOT / "artifacts" / "reports"
REGISTRY_PATH = MODEL_ROOT / "registry.json"

In [2]:
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
active_version = registry["active_model_version"]
active_entry = next(item for item in registry["models"] if item["model_version"] == active_version)
artifact_path = Path(active_entry["artifact_path"])
metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
manifest = FeatureManifest(**metadata["extra_metadata"]["feature_manifest"])
scaler = ScalerStore.load(MODEL_ROOT / "scaler_params.joblib")
roads = pd.read_csv(config.paths.roads_csv)
traffic = pd.read_csv(config.paths.traffic_csv)

print(f"Active model: {active_version}")
print(f"Lookback: {manifest.lookback}, horizon: {manifest.horizon}, features: {len(manifest.feature_columns)}")
print(f"Roads: {len(roads)}")

Active model: lstm-real-20260518-032721
Lookback: 12, horizon: 4, features: 41
Roads: 50


In [3]:
@dataclass(frozen=True)
class NotebookRealtimePrediction:
    road_id: str
    horizon_minutes: int
    predicted_speed: float
    congestion_level: str
    confidence_score: float
    uncertainty_lower: float
    uncertainty_upper: float
    response_time_ms: float
    prediction_method: str
    degraded: bool
    model_version: str
    data_quality: dict
    feature_quality: dict


class NotebookLSTMRunner:
    def __init__(self, artifact_path: Path, scaler_store: ScalerStore, device: str = "cpu") -> None:
        self.artifact_path = artifact_path
        self.scaler_store = scaler_store
        self.device = torch.device(device)
        self.metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
        model_config = dict(self.metadata["model_config"])
        model_config["hidden_sizes"] = tuple(model_config["hidden_sizes"])
        self.model = TrafficLSTM(LSTMModelConfig(**model_config)).to(self.device)
        checkpoint = torch.load(self.metadata["checkpoint_path"], map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

    def predict_kmh(self, sequence: np.ndarray) -> np.ndarray:
        tensor = torch.as_tensor(sequence, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            scaled = self.model(tensor).detach().cpu().numpy()
        restored = self.scaler_store.inverse_transform_speed(scaled.reshape(-1, 1)).reshape(scaled.shape)
        return np.clip(restored, 0.0, 120.0)


def build_live_buffer(traffic: pd.DataFrame, *, lookback: int, max_timesteps: int = 48) -> tuple[LiveBufferManager, pd.Timestamp]:
    required = ["road_id", "current_speed", "confidence", "collected_at_wib"]
    data = traffic[required].copy()
    data["collected_at_wib"] = pd.to_datetime(data["collected_at_wib"], errors="coerce")
    data = data.dropna(subset=required).sort_values(["road_id", "collected_at_wib"])
    latest = data.groupby("road_id", group_keys=False).tail(max(lookback, max_timesteps))
    manager = LiveBufferManager(min_timesteps=lookback, max_timesteps=max_timesteps)
    records = []
    max_timestamp = pd.Timestamp(latest["collected_at_wib"].max())
    if max_timestamp.tzinfo is None:
        max_timestamp = max_timestamp.tz_localize(config.data.timezone)
    else:
        max_timestamp = max_timestamp.tz_convert(config.data.timezone)
    for row in latest.itertuples(index=False):
        timestamp = pd.Timestamp(row.collected_at_wib)
        if timestamp.tzinfo is None:
            timestamp = timestamp.tz_localize(config.data.timezone)
        else:
            timestamp = timestamp.tz_convert(config.data.timezone)
        records.append(
            LiveTrafficRecord(
                road_id=str(row.road_id),
                current_speed=float(row.current_speed),
                confidence=float(row.confidence),
                timestamp=timestamp.to_pydatetime(),
                freshness_indicator=max_timestamp.to_pydatetime() - timestamp.to_pydatetime(),
            )
        )
    manager.append_many(records)
    return manager, max_timestamp


live_buffer, now = build_live_buffer(traffic, lookback=manifest.lookback)
engineer = OnlineFeatureEngineer(manifest=manifest, buffer_manager=live_buffer, roads=roads, scaler_store=scaler)
runner = NotebookLSTMRunner(artifact_path=artifact_path, scaler_store=scaler)
quality_monitor = DataQualityMonitor()
confidence_adjuster = ConfidenceAdjuster()
print(f"Buffered roads: {len(live_buffer.buffers)}; now={now.isoformat()}")

Buffered roads: 50; now=2026-04-30T03:45:01+07:00

In [4]:
def predict_realtime(road_id: str, horizon_minutes: int = 60, response_time_target_ms: float = 1000.0) -> NotebookRealtimePrediction:
    started = time.perf_counter()
    latest_records = [records[-1] for rid in sorted(live_buffer.buffers) if (records := live_buffer.get_latest(rid))]
    expected_roads = set(roads["road_id"].astype(str))
    quality_report = quality_monitor.evaluate(latest_records, expected_road_ids=expected_roads, now=now.to_pydatetime())
    feature_result = engineer.build_for_road(road_id)
    prediction_kmh = runner.predict_kmh(feature_result.sequence).reshape(-1)
    horizon_index = {15: 0, 30: 1, 45: 2, 60: 3}[horizon_minutes]
    predicted_speed = float(prediction_kmh[horizon_index])
    uncertainty_margin = max(float(np.std(prediction_kmh)), 1.0)
    source_confidence = float(np.mean([record.confidence for record in live_buffer.get_latest(road_id, manifest.lookback)]))
    confidence = confidence_adjuster.adjust(
        base_confidence=0.90,
        uncertainty_margin=uncertainty_margin,
        source_confidence=source_confidence,
        data_quality=quality_report,
    )
    road = roads[roads["road_id"].astype(str) == str(road_id)].iloc[0]
    free_flow_speed = float(road.get("free_flow_speed", max(predicted_speed, 1.0)) or max(predicted_speed, 1.0))
    response_time_ms = (time.perf_counter() - started) * 1000
    degraded = feature_result.quality.status != "healthy" or quality_report.status != "healthy"
    result = NotebookRealtimePrediction(
        road_id=road_id,
        horizon_minutes=horizon_minutes,
        predicted_speed=round(predicted_speed, 3),
        congestion_level=classify_congestion(predicted_speed, free_flow_speed),
        confidence_score=round(confidence.adjusted_confidence, 6),
        uncertainty_lower=round(max(0.0, predicted_speed - uncertainty_margin), 3),
        uncertainty_upper=round(min(120.0, predicted_speed + uncertainty_margin), 3),
        response_time_ms=round(response_time_ms, 3),
        prediction_method="live_lstm_notebook_engine",
        degraded=degraded,
        model_version=active_version,
        data_quality={**asdict(quality_report), "timestamp": quality_report.timestamp.isoformat()},
        feature_quality=feature_result.quality.to_dict(),
    )
    if response_time_ms > response_time_target_ms:
        raise RuntimeError(f"Response time target exceeded: {response_time_ms:.3f} ms > {response_time_target_ms:.3f} ms")
    return result


prediction = predict_realtime("SBM_BHY_01", horizon_minutes=60)
prediction

NotebookRealtimePrediction(road_id='SBM_BHY_01', horizon_minutes=60, predicted_speed=29.29, congestion_level='free_flow', confidence_score=0.8775, uncertainty_lower=28.29, uncertainty_upper=30.29, response_time_ms=100.468, prediction_method='live_lstm_notebook_engine', degraded=False, model_version='lstm-real-20260518-032721', data_quality={'timestamp': '2026-04-30T03:45:01+07:00', 'completeness': 1.0, 'average_confidence': 1.0, 'stale_roads': [], 'missing_roads': [], 'delayed_roads': [], 'low_confidence_roads': [], 'outlier_roads': [], 'api_uptime': 1.0, 'fallback_recommendation': 'use_live_lstm', 'quality_issues': {'missing_roads': 0, 'stale_roads': 0, 'delayed_roads': 0, 'low_confidence_roads': 0, 'outlier_roads': 0, 'api_failures': 0}, 'status': 'healthy'}, feature_quality={'road_id': 'SBM_BHY_01', 'observed_timesteps': 12, 'required_timesteps': 12, 'has_minimum_history': True, 'missing_feature_columns': [], 'padded_timesteps': 0, 'status': 'healthy'})

In [5]:
batch_started = time.perf_counter()
batch_results = [predict_realtime(road_id, horizon_minutes=60) for road_id in roads["road_id"].astype(str).head(5)]
batch_response_time_ms = (time.perf_counter() - batch_started) * 1000

checks = {
    "live_buffer_has_roads": len(live_buffer.buffers) == len(roads),
    "feature_sequence_shape_valid": engineer.build_for_road("SBM_BHY_01").sequence.shape == (1, manifest.lookback, len(manifest.feature_columns)),
    "single_prediction_within_bounds": 0.0 <= prediction.predicted_speed <= 120.0,
    "confidence_within_bounds": 0.0 <= prediction.confidence_score <= 1.0,
    "single_response_under_target": prediction.response_time_ms <= 1000.0,
    "batch_prediction_count_valid": len(batch_results) == 5,
    "batch_response_under_target": batch_response_time_ms <= 5000.0,
}
if not all(checks.values()):
    raise RuntimeError(checks)

report = {
    "status": "passed",
    "model_version": active_version,
    "runner_location": "notebooks/08_realtime_inference_engine_validation.ipynb",
    "road_id": prediction.road_id,
    "horizon_minutes": prediction.horizon_minutes,
    "predicted_speed": prediction.predicted_speed,
    "congestion_level": prediction.congestion_level,
    "confidence_score": prediction.confidence_score,
    "uncertainty_lower": prediction.uncertainty_lower,
    "uncertainty_upper": prediction.uncertainty_upper,
    "single_response_time_ms": prediction.response_time_ms,
    "batch_response_time_ms": round(batch_response_time_ms, 3),
    "prediction_method": prediction.prediction_method,
    "degraded": prediction.degraded,
    "feature_quality": prediction.feature_quality,
    "data_quality": prediction.data_quality,
    "batch_predictions": [asdict(item) for item in batch_results],
    "checks": checks,
}

report_path = REPORT_ROOT / active_version / "realtime_inference_engine_validation.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps({"report": str(report_path), "prediction": prediction.predicted_speed, "response_ms": prediction.response_time_ms}, indent=2))

{
  "report": "C:\\AIproject\\AWAI\\artifacts\\reports\\lstm-real-20260518-032721\\realtime_inference_engine_validation.json",
  "prediction": 29.29,
  "response_ms": 100.468
}
